In [642]:
import pandas as pd 
import pathlib as Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
import seaborn as sns
import matplotlib.pyplot as plt

In [605]:
ruta_archivo = "../data/raw/vista_modelo"

In [606]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

In [638]:
scaler = MinMaxScaler()

In [607]:
df = pd.read_csv(ruta_archivo, sep=",", encoding="latin-1", low_memory=False)

In [608]:
df

,punt_global,estrato,personas_hogar,cuartos_hogar,edu_madre,edu_padre,tiene_internet,tiene_computador,tiene_automovil,tiene_lavadora,cole_naturaleza,cole_area,cole_jornada,cole_bilingue,cole_caracter,cole_depto
0,170,1,Seis,Tres,2.0,2.0,0.0,0.0,0.0,0.0,OFICIAL,URBANO,MAÃANA,N,TÃCNICO/ACADÃMICO,META
1,248,5,3 a 4,Tres,2.0,1.0,1.0,1.0,1.0,1.0,OFICIAL,URBANO,UNICA,N,TÃCNICO/ACADÃMICO,NORTE SANTANDER
2,248,5,3 a 4,Tres,2.0,1.0,1.0,1.0,1.0,1.0,OFICIAL,URBANO,UNICA,N,TÃCNICO/ACADÃMICO,NORTE SANTANDER
3,278,2,7 a 8,Tres,3.0,3.0,1.0,0.0,0.0,0.0,OFICIAL,URBANO,UNICA,N,TÃCNICO/ACADÃMICO,MAGDALENA
4,305,2,5 a 6,Tres,2.0,1.0,1.0,0.0,0.0,1.0,OFICIAL,URBANO,MAÃANA,N,TÃCNICO/ACADÃMICO,NORTE SANTANDER
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163865,235,1,Ocho,Cuatro,1.0,2.0,0.0,0.0,0.0,0.0,OFICIAL,RURAL,TARDE,N,ACADÃMICO,CORDOBA
163866,191,1,Cuatro,Dos,2.0,1.0,1.0,1.0,0.0,0.0,NO OFICIAL,URBANO,NOCHE,NaN,NaN,NORTE SANTANDER
163867,189,1,7 a 8,Tres,2.0,2.0,0.0,0.0,0.0,1.0,OFICIAL,URBANO,MAÃANA,N,TÃCNICO,BOLIVAR
163868,198,1,Tres,Tres,2.0,2.0,0.0,1.0,0.0,1.0,NO OFICIAL,URBANO,COMPLETA,NaN,ACADÃMICO,TOLIMA


In [609]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 163870 entries, 0 to 163869
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   punt_global       163870 non-null  int64  
 1   estrato           163870 non-null  int64  
 2   personas_hogar    162448 non-null  object 
 3   cuartos_hogar     162562 non-null  object 
 4   edu_madre         160940 non-null  float64
 5   edu_padre         153886 non-null  float64
 6   tiene_internet    163498 non-null  float64
 7   tiene_computador  162622 non-null  float64
 8   tiene_automovil   162487 non-null  float64
 9   tiene_lavadora    162701 non-null  float64
 10  cole_naturaleza   163870 non-null  object 
 11  cole_area         163870 non-null  object 
 12  cole_jornada      163870 non-null  object 
 13  cole_bilingue     141011 non-null  object 
 14  cole_caracter     161332 non-null  object 
 15  cole_depto        163870 non-null  object 
dtypes: float64(6), int64

In [610]:
porcentaje_nulos = (df.isnull().sum() / len(df)) * 100

print(porcentaje_nulos.sort_values(ascending=False))


cole_bilingue       13.949472
edu_padre            6.092634
edu_madre            1.788003
cole_caracter        1.548789
personas_hogar       0.867761
tiene_automovil      0.843962
cuartos_hogar        0.798194
tiene_computador     0.761579
tiene_lavadora       0.713370
tiene_internet       0.227009
estrato              0.000000
punt_global          0.000000
cole_area            0.000000
cole_naturaleza      0.000000
cole_jornada         0.000000
cole_depto           0.000000
dtype: float64


In [611]:
df['personas_hogar'].value_counts()

personas_hogar
3 a 4          47541
5 a 6          30026
Cuatro         20560
Cinco          15861
Tres           11972
7 a 8           7695
Seis            7648
1 a 2           6876
Dos             3578
Siete           3575
9 o mÃ¡s        2841
Ocho            1887
Nueve            837
Diez             522
Doce o mÃ¡s      401
Una              391
Once             237
Name: count, dtype: int64

In [612]:
def unificar_personas(x):
    if pd.isna(x):
        return 'Ninguna'
    x = x.strip().lower()
    if x in ('una', 'dos', '1 a 2'):
        return '1 a 2'
    if x in ('tres', 'cuatro', '3 a 4'):
        return '3 a 4'
    if x in ('cinco', 'seis', '5 a 6'):
        return '5 a 6'
    if x in ('siete', 'ocho', '7 a 8'):
        return '7 a 8'
    if x in ('nueve', 'diez', 'once') or x.startswith('9 o') or x.startswith('doce'):
        return '9 o más'
    return None

df['personas_hogar'] = df['personas_hogar'].map(unificar_personas)
df['personas_hogar'].value_counts(dropna=False)

personas_hogar
3 a 4      80073
5 a 6      53535
7 a 8      13157
1 a 2      10845
9 o más     4838
Ninguna     1422
Name: count, dtype: int64

In [613]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 163870 entries, 0 to 163869
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   punt_global       163870 non-null  int64  
 1   estrato           163870 non-null  int64  
 2   personas_hogar    163870 non-null  object 
 3   cuartos_hogar     162562 non-null  object 
 4   edu_madre         160940 non-null  float64
 5   edu_padre         153886 non-null  float64
 6   tiene_internet    163498 non-null  float64
 7   tiene_computador  162622 non-null  float64
 8   tiene_automovil   162487 non-null  float64
 9   tiene_lavadora    162701 non-null  float64
 10  cole_naturaleza   163870 non-null  object 
 11  cole_area         163870 non-null  object 
 12  cole_jornada      163870 non-null  object 
 13  cole_bilingue     141011 non-null  object 
 14  cole_caracter     161332 non-null  object 
 15  cole_depto        163870 non-null  object 
dtypes: float64(6), int64

In [614]:
df['personas_hogar'].value_counts()

personas_hogar
3 a 4      80073
5 a 6      53535
7 a 8      13157
1 a 2      10845
9 o más     4838
Ninguna     1422
Name: count, dtype: int64

In [615]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 163870 entries, 0 to 163869
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   punt_global       163870 non-null  int64  
 1   estrato           163870 non-null  int64  
 2   personas_hogar    163870 non-null  object 
 3   cuartos_hogar     162562 non-null  object 
 4   edu_madre         160940 non-null  float64
 5   edu_padre         153886 non-null  float64
 6   tiene_internet    163498 non-null  float64
 7   tiene_computador  162622 non-null  float64
 8   tiene_automovil   162487 non-null  float64
 9   tiene_lavadora    162701 non-null  float64
 10  cole_naturaleza   163870 non-null  object 
 11  cole_area         163870 non-null  object 
 12  cole_jornada      163870 non-null  object 
 13  cole_bilingue     141011 non-null  object 
 14  cole_caracter     161332 non-null  object 
 15  cole_depto        163870 non-null  object 
dtypes: float64(6), int64

In [616]:
df['tiene_internet'].value_counts()

tiene_internet
1.0    97180
0.0    66318
Name: count, dtype: int64

In [617]:
df['tiene_lavadora'].value_counts()

tiene_lavadora
1.0    120477
0.0     42224
Name: count, dtype: int64

In [618]:
df['cuartos_hogar'].value_counts()

cuartos_hogar
Tres           66901
Dos            57575
Cuatro         21700
Uno             6941
Cinco           6251
Seis o mas      1938
Seis             766
Siete            251
Ocho             121
Diez o mÃ¡s       69
Nueve             49
Name: count, dtype: int64

In [619]:
def unificar_cuartos(x):
    if pd.isna(x):
        return 'Ninguno'
    x = x.strip().lower()
    if x in ('uno', 'dos', '1 a 2'):
        return '1 a 2'
    if x in ('tres', 'cuatro', '3 a 4'):
        return '3 a 4'
    if x in ('cinco', 'seis', '5 a 6'):
        return '5 a 6'
    if x in ('siete', 'ocho', '7 a 8') or x.startswith('seis o'):
        return '7 a 8'
    if x in ('nueve', 'diez', 'once', ) or x.startswith('diez o'):
        return '9 o más'
    return None

df['cuartos_hogar'] = df['cuartos_hogar'].map(unificar_cuartos)
df['cuartos_hogar'].value_counts(dropna=False)

cuartos_hogar
3 a 4      88601
1 a 2      64516
5 a 6       7017
7 a 8       2310
Ninguno     1308
9 o más      118
Name: count, dtype: int64

In [620]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 163870 entries, 0 to 163869
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   punt_global       163870 non-null  int64  
 1   estrato           163870 non-null  int64  
 2   personas_hogar    163870 non-null  object 
 3   cuartos_hogar     163870 non-null  object 
 4   edu_madre         160940 non-null  float64
 5   edu_padre         153886 non-null  float64
 6   tiene_internet    163498 non-null  float64
 7   tiene_computador  162622 non-null  float64
 8   tiene_automovil   162487 non-null  float64
 9   tiene_lavadora    162701 non-null  float64
 10  cole_naturaleza   163870 non-null  object 
 11  cole_area         163870 non-null  object 
 12  cole_jornada      163870 non-null  object 
 13  cole_bilingue     141011 non-null  object 
 14  cole_caracter     161332 non-null  object 
 15  cole_depto        163870 non-null  object 
dtypes: float64(6), int64

In [621]:
df['cole_caracter'].value_counts()

cole_caracter
ACADÃMICO             88492
TÃCNICO/ACADÃMICO    53946
TÃCNICO               17663
NO APLICA               1231
Name: count, dtype: int64

In [622]:
df['cole_caracter'].isna().sum()

np.int64(2538)

In [623]:
df['edu_madre'].value_counts()

edu_madre
2.0    69228
1.0    43456
4.0    21235
3.0    19515
5.0     4393
0.0     3113
Name: count, dtype: int64

In [624]:
df = df.dropna(subset=['tiene_internet', 'tiene_lavadora','tiene_computador', 'tiene_automovil', 'cole_caracter', 'edu_madre', 'edu_padre' ])

In [625]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 148878 entries, 0 to 163869
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   punt_global       148878 non-null  int64  
 1   estrato           148878 non-null  int64  
 2   personas_hogar    148878 non-null  object 
 3   cuartos_hogar     148878 non-null  object 
 4   edu_madre         148878 non-null  float64
 5   edu_padre         148878 non-null  float64
 6   tiene_internet    148878 non-null  float64
 7   tiene_computador  148878 non-null  float64
 8   tiene_automovil   148878 non-null  float64
 9   tiene_lavadora    148878 non-null  float64
 10  cole_naturaleza   148878 non-null  object 
 11  cole_area         148878 non-null  object 
 12  cole_jornada      148878 non-null  object 
 13  cole_bilingue     129770 non-null  object 
 14  cole_caracter     148878 non-null  object 
 15  cole_depto        148878 non-null  object 
dtypes: float64(6), int64(2), 

In [626]:
df['cole_depto'].value_counts()

cole_depto
ANTIOQUIA          19266
BOGOTA             16095
VALLE              13152
CUNDINAMARCA        9981
BOGOTÃ             8452
ATLANTICO           8394
SANTANDER           6994
BOLIVAR             6492
CORDOBA             5356
BOYACA              4724
NARIÃO             4552
TOLIMA              4510
NORTE SANTANDER     4333
MAGDALENA           4115
CAUCA               3763
HUILA               3676
CESAR               3338
META                3102
SUCRE               2916
CALDAS              2871
RISARALDA           2808
LA GUAJIRA          1858
QUINDIO             1821
CASANARE            1468
CHOCO               1093
CAQUETA             1036
PUTUMAYO            1009
ARAUCA               823
GUAVIARE             239
AMAZONAS             210
SAN ANDRES           180
VICHADA              111
VAUPES                79
GUAINIA               61
Name: count, dtype: int64

In [627]:
def normalizacion(x):
    if x =='BOGOTÃ':
        return 'BOGOTA'
    return x

df['cole_depto'] = df['cole_depto'].map(normalizacion)
df['cole_depto'].value_counts(dropna=False)

C:\Users\dalz2\AppData\Local\Temp\ipykernel_1044\2808514159.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['cole_depto'] = df['cole_depto'].map(normalizacion)


cole_depto
BOGOTA             24547
ANTIOQUIA          19266
VALLE              13152
CUNDINAMARCA        9981
ATLANTICO           8394
SANTANDER           6994
BOLIVAR             6492
CORDOBA             5356
BOYACA              4724
NARIÃO             4552
TOLIMA              4510
NORTE SANTANDER     4333
MAGDALENA           4115
CAUCA               3763
HUILA               3676
CESAR               3338
META                3102
SUCRE               2916
CALDAS              2871
RISARALDA           2808
LA GUAJIRA          1858
QUINDIO             1821
CASANARE            1468
CHOCO               1093
CAQUETA             1036
PUTUMAYO            1009
ARAUCA               823
GUAVIARE             239
AMAZONAS             210
SAN ANDRES           180
VICHADA              111
VAUPES                79
GUAINIA               61
Name: count, dtype: int64

In [628]:
df['cole_bilingue'].value_counts()

cole_bilingue
N    126810
S      2960
Name: count, dtype: int64

In [629]:
df['cole_bilingue'].isna().sum()

np.int64(19108)

In [630]:
df = df.fillna({ 'cole_bilingue': 'DESCONOCIDO' })

In [631]:
df['cole_bilingue'].value_counts()

cole_bilingue
N              126810
DESCONOCIDO     19108
S                2960
Name: count, dtype: int64

In [632]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 148878 entries, 0 to 163869
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   punt_global       148878 non-null  int64  
 1   estrato           148878 non-null  int64  
 2   personas_hogar    148878 non-null  object 
 3   cuartos_hogar     148878 non-null  object 
 4   edu_madre         148878 non-null  float64
 5   edu_padre         148878 non-null  float64
 6   tiene_internet    148878 non-null  float64
 7   tiene_computador  148878 non-null  float64
 8   tiene_automovil   148878 non-null  float64
 9   tiene_lavadora    148878 non-null  float64
 10  cole_naturaleza   148878 non-null  object 
 11  cole_area         148878 non-null  object 
 12  cole_jornada      148878 non-null  object 
 13  cole_bilingue     148878 non-null  object 
 14  cole_caracter     148878 non-null  object 
 15  cole_depto        148878 non-null  object 
dtypes: float64(6), int64(2), 

In [633]:
X = df.drop(columns=['punt_global'])
y = df['punt_global']

In [634]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 148878 entries, 0 to 163869
Data columns (total 15 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   estrato           148878 non-null  int64  
 1   personas_hogar    148878 non-null  object 
 2   cuartos_hogar     148878 non-null  object 
 3   edu_madre         148878 non-null  float64
 4   edu_padre         148878 non-null  float64
 5   tiene_internet    148878 non-null  float64
 6   tiene_computador  148878 non-null  float64
 7   tiene_automovil   148878 non-null  float64
 8   tiene_lavadora    148878 non-null  float64
 9   cole_naturaleza   148878 non-null  object 
 10  cole_area         148878 non-null  object 
 11  cole_jornada      148878 non-null  object 
 12  cole_bilingue     148878 non-null  object 
 13  cole_caracter     148878 non-null  object 
 14  cole_depto        148878 non-null  object 
dtypes: float64(6), int64(1), object(8)
memory usage: 18.2+ MB


In [635]:
encoded_data = encoder.fit_transform(X[['personas_hogar', 'cuartos_hogar', 'cole_naturaleza', 'cole_area','cole_jornada', 'cole_bilingue', 'cole_caracter', 'cole_depto']])

In [636]:
X = pd.DataFrame(
    encoded_data, 
    columns=encoder.get_feature_names_out(['personas_hogar', 'cuartos_hogar', 'cole_naturaleza', 'cole_area','cole_jornada', 'cole_bilingue', 'cole_caracter', 'cole_depto'])
)

In [637]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148878 entries, 0 to 148877
Data columns (total 62 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   personas_hogar_1 a 2               148878 non-null  float64
 1   personas_hogar_3 a 4               148878 non-null  float64
 2   personas_hogar_5 a 6               148878 non-null  float64
 3   personas_hogar_7 a 8               148878 non-null  float64
 4   personas_hogar_9 o más             148878 non-null  float64
 5   personas_hogar_Ninguna             148878 non-null  float64
 6   cuartos_hogar_1 a 2                148878 non-null  float64
 7   cuartos_hogar_3 a 4                148878 non-null  float64
 8   cuartos_hogar_5 a 6                148878 non-null  float64
 9   cuartos_hogar_7 a 8                148878 non-null  float64
 10  cuartos_hogar_9 o más              148878 non-null  float64
 11  cuartos_hogar_Ninguno              1488